In [ ]:
import spacy
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd

# Load pre-trained NER model
nlp = spacy.load("en_core_web_sm")

In [ ]:
# Real-world text samples
texts = [
    "Apple Inc. is planning to open a new store in New York City next month. CEO Tim Cook announced this during a press conference.",
    "Elon Musk's Tesla delivered over 500,000 vehicles in 2020. The company's headquarters is located in California.",
    "Microsoft announced that Satya Nadella will speak at the conference in Seattle on December 15, 2024.",
    "Amazon's founder Jeff Bezos stepped down as CEO. The company operates in over 15 countries including India and Japan.",
    "Google's parent company Alphabet reported strong earnings. Sundar Pichai praised the team's work in Mountain View."
]

# Process texts with NER
print("Extracting Named Entities:\n")
for i, text in enumerate(texts, 1):
    doc = nlp(text)
    print(f"Text {i}: {text}")
    print("Entities:")
    for ent in doc.ents:
        print(f"  - {ent.text}: {ent.label_}")
    print()

In [ ]:
# Ground truth annotations for evaluation
ground_truth = [
    {"text": "Apple Inc. is planning to open a new store in New York City next month. CEO Tim Cook announced this during a press conference.",
     "entities": [("Apple Inc.", "ORG"), ("New York City", "GPE"), ("next month", "DATE"), ("Tim Cook", "PERSON")]},
    
    {"text": "Elon Musk's Tesla delivered over 500,000 vehicles in 2020. The company's headquarters is located in California.",
     "entities": [("Elon Musk", "PERSON"), ("Tesla", "ORG"), ("500,000", "CARDINAL"), ("2020", "DATE"), ("California", "GPE")]},
    
    {"text": "Microsoft announced that Satya Nadella will speak at the conference in Seattle on December 15, 2024.",
     "entities": [("Microsoft", "ORG"), ("Satya Nadella", "PERSON"), ("Seattle", "GPE"), ("December 15, 2024", "DATE")]},
    
    {"text": "Amazon's founder Jeff Bezos stepped down as CEO. The company operates in over 15 countries including India and Japan.",
     "entities": [("Amazon", "ORG"), ("Jeff Bezos", "PERSON"), ("15", "CARDINAL"), ("India", "GPE"), ("Japan", "GPE")]},
    
    {"text": "Google's parent company Alphabet reported strong earnings. Sundar Pichai praised the team's work in Mountain View.",
     "entities": [("Google", "ORG"), ("Alphabet", "ORG"), ("Sundar Pichai", "PERSON"), ("Mountain View", "GPE")]}
]

In [ ]:
# Extract predictions from NER model
def extract_entities(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

predictions = []
true_labels = []

for item in ground_truth:
    text = item["text"]
    true_ents = set(item["entities"])
    pred_ents = set(extract_entities(text))
    
    predictions.append(pred_ents)
    true_labels.append(true_ents)

print("Ground Truth vs Predictions:\n")
for i, (true, pred) in enumerate(zip(true_labels, predictions), 1):
    print(f"Text {i}:")
    print(f"  True: {true}")
    print(f"  Predicted: {pred}")
    print()

In [ ]:
# Calculate metrics
def calculate_metrics(true_labels, predictions):
    true_positives = 0
    false_positives = 0
    false_negatives = 0
    
    for true, pred in zip(true_labels, predictions):
        true_positives += len(true & pred)
        false_positives += len(pred - true)
        false_negatives += len(true - pred)
    
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    total_predictions = sum(len(pred) for pred in predictions)
    total_true = sum(len(true) for true in true_labels)
    accuracy = true_positives / max(total_predictions, total_true) if max(total_predictions, total_true) > 0 else 0
    
    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1_score,
        "True Positives": true_positives,
        "False Positives": false_positives,
        "False Negatives": false_negatives
    }

metrics = calculate_metrics(true_labels, predictions)

print("NER System Performance Metrics:\n")
print("="*50)
for metric, value in metrics.items():
    if isinstance(value, float):
        print(f"{metric:20s}: {value:.4f}")
    else:
        print(f"{metric:20s}: {value}")
print("="*50)

In [ ]:
# Detailed entity-wise analysis
entity_stats = {}

for item in ground_truth:
    text = item["text"]
    pred_ents = extract_entities(text)
    true_ents = item["entities"]
    
    for ent_text, ent_label in true_ents:
        if ent_label not in entity_stats:
            entity_stats[ent_label] = {"correct": 0, "total": 0}
        entity_stats[ent_label]["total"] += 1
        
        if (ent_text, ent_label) in pred_ents:
            entity_stats[ent_label]["correct"] += 1

print("\nEntity Type Analysis:")
print("="*50)
for ent_type, stats in sorted(entity_stats.items()):
    accuracy = stats["correct"] / stats["total"] if stats["total"] > 0 else 0
    print(f"{ent_type:15s}: {stats['correct']}/{stats['total']} correct ({accuracy:.2%})")
print("="*50)

In [8]:
# Visualize results
results_df = pd.DataFrame([metrics])
print("\nResults Summary:")
print(results_df.to_string(index=False))


Results Summary:
 Accuracy  Precision   Recall  F1-Score  True Positives  False Positives  False Negatives
 0.863636   0.904762 0.863636  0.883721              19                2                3
